In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
df = pd.read_csv('/content/processed_nlp_data.csv')

df.head()

,Customer Age,Customer Gender,Product Purchased,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,...,purchase_day,desc_length,subject_length,cleaned_description,cleaned_subject,final_description,final_subject,lemmatized_description,lemmatized_subject,final_text
0,32,Other,GoPro Hero,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,...,22,284,13,im having an issue with the productpurchased p...,product setup,im issue productpurchased please assist billin...,product setup,im issue productpurchased please assist billin...,product setup,product setup im issue productpurchased please...
1,42,Female,LG Smart TV,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,...,22,282,24,im having an issue with the productpurchased p...,peripheral compatibility,im issue productpurchased please assist need c...,peripheral compatibility,im issue productpurchased please assist need c...,peripheral compatibility,peripheral compatibility im issue productpurch...
2,48,Other,Dell XPS,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,...,14,275,15,im facing a problem with my productpurchased t...,network problem,im facing problem productpurchased productpurc...,network problem,im facing problem productpurchased productpurc...,network problem,network problem im facing problem productpurch...
3,27,Female,Microsoft Office,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,...,13,262,14,im having an issue with the productpurchased p...,account access,im issue productpurchased please assist proble...,account access,im issue productpurchased please assist proble...,account access,account access im issue productpurchased pleas...
4,67,Female,Autodesk AutoCAD,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,...,4,333,9,im having an issue with the productpurchased p...,data loss,im issue productpurchased please assist note s...,data loss,im issue productpurchased please assist note s...,data loss,data loss im issue productpurchased please ass...


In [3]:
X = df['final_text']
y = df['Ticket Priority']   # better than Ticket Type

In [4]:
le = LabelEncoder()
y = le.fit_transform(y)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [6]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [7]:
X_train_pad = pad_sequences(X_train_seq, maxlen=100)
X_test_pad = pad_sequences(X_test_seq, maxlen=100)

In [8]:
model = Sequential()

model.add(Embedding(input_dim=5000, output_dim=64, input_length=100))
model.add(Bidirectional(LSTM(64)))
model.add(Dense(32, activation='relu'))
model.add(Dense(4, activation='softmax'))   # 4 classes (priority)

model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test_pad, y_test)
)

Epoch 1/5
212/212 ━━━━━━━━━━━━━━━━━━━━ 36s 150ms/step - accuracy: 0.2546 - loss: 1.3870 - val_accuracy: 0.2473 - val_loss: 1.3875
Epoch 2/5
212/212 ━━━━━━━━━━━━━━━━━━━━ 28s 130ms/step - accuracy: 0.2623 - loss: 1.3842 - val_accuracy: 0.2462 - val_loss: 1.3907
Epoch 3/5
212/212 ━━━━━━━━━━━━━━━━━━━━ 38s 117ms/step - accuracy: 0.2951 - loss: 1.3704 - val_accuracy: 0.2332 - val_loss: 1.4202
Epoch 4/5
212/212 ━━━━━━━━━━━━━━━━━━━━ 24s 113ms/step - accuracy: 0.3808 - loss: 1.3071 - val_accuracy: 0.2414 - val_loss: 1.5351
Epoch 5/5
212/212 ━━━━━━━━━━━━━━━━━━━━ 44s 128ms/step - accuracy: 0.4806 - loss: 1.1825 - val_accuracy: 0.2332 - val_loss: 1.6645


In [10]:
loss, accuracy = model.evaluate(X_test_pad, y_test)

print("BiLSTM Accuracy:", accuracy)

53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.2332 - loss: 1.6645
BiLSTM Accuracy: 0.23317591845989227


## 🔍 Model Comparison

| Model                  | Accuracy |
|----------------------|---------|
| Logistic Regression   | ~18%    |
| Naive Bayes           | ~20%    |
| Random Forest         | ~24%    |
| XGBoost               | ~23%    |
| BiLSTM (Deep Learning)| ~23%    |

---

### 📌 Key Insight

- All models perform around random chance (~20%)
- Indicates lack of meaningful signal in dataset
- Increasing model complexity does not improve results

---

### 🎯 Final Conclusion

This experiment demonstrates that:

➡️ Data quality is more important than model complexity  
➡️ Even advanced models like BiLSTM cannot perform well on poor-quality datasets  
➡️ Proper feature engineering and real-world data are crucial for good performance